In [115]:
import json
import requests
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain

from pathlib import Path

EMB_DIR = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FILE = EMB_DIR / "cicids_sample.csv"
EMBED_FILE = EMB_DIR / "cicids_embeddings.npy"

COMMUNITY_FILE = OUTPUT_DIR / "community_assignments.csv"
SUMMARY_FILE = OUTPUT_DIR / "community_summary.csv"
CLUSTER_METRICS_FILE = OUTPUT_DIR / "clustering_metrics.json"
TRIPLES_FILE = OUTPUT_DIR / "community_triples.json"
TRIPLE_METRICS_FILE = OUTPUT_DIR / "triple_metrics.json"

SIMILARITY_THRESHOLD = 0.90
MIN_COMMUNITY_SIZE = 3
MAX_ALERTS_PER_COMMUNITY = 10

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:3b"
TIMEOUT = 300

In [116]:
# Load sample and embeddings
sample = pd.read_csv(SAMPLE_FILE, index_col="sample_id", low_memory=False)
embeddings = np.load(EMBED_FILE)

print("Sample shape:", sample.shape)
print("Embeddings shape:", embeddings.shape)

assert len(sample) == len(embeddings)

Sample shape: (10673, 16)
Embeddings shape: (10673, 384)


In [117]:
# Build cosine similarity graph
similarity_matrix = np.dot(embeddings, embeddings.T)

G = nx.Graph()
G.add_nodes_from(sample.index.tolist())

for i in range(len(sample)):
    for j in range(i + 1, len(sample)):
        sim = similarity_matrix[i, j]
        if sim >= SIMILARITY_THRESHOLD:
            G.add_edge(i, j, weight=float(sim))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 10673
Edges: 6837559


In [118]:
# Run Louvain community detection
partition = community_louvain.best_partition(G, weight="weight")

community_df = sample.reset_index().copy()
community_df["community_id"] = community_df["sample_id"].map(partition)

community_df = community_df.dropna(subset=["community_id"]).copy()
community_df["community_id"] = community_df["community_id"].astype(int)

# Remove tiny communities if needed
community_sizes = community_df["community_id"].value_counts()
valid_communities = community_sizes[community_sizes >= MIN_COMMUNITY_SIZE].index
community_df = community_df[community_df["community_id"].isin(valid_communities)].copy()

community_df.to_csv(COMMUNITY_FILE, index=False)
print(f"Saved to {COMMUNITY_FILE}")

n_communities = community_df["community_id"].nunique()
print(f"Number of communities: {n_communities}")

Saved to ../data/results/community_assignments.csv
Number of communities: 16


In [119]:
#community_df.head()

In [120]:
# inspect a few communities
for community_id, group in community_df.groupby("community_id"):
    print(f"\nCommunity {community_id} | size={len(group)}")
    print("-" * 60)
    for text in group["alert_text"].head(5):
        print(text)
    if community_id >= 2:
        break


Community 0 | size=3188
------------------------------------------------------------
Network flow to unknown service on port 1123. very short connection. low packet count. Duration 38 microseconds. Forward packets 1. Backward packets 1. Flow bytes per second 210526.32. TCP flags: none.
Network flow to unknown service on port 1580. very short connection. Duration 50 microseconds. Forward packets 2. Backward packets 2. Flow bytes per second 320000.0. TCP flags: none.
Network flow to unknown service on port 425. very short connection. low packet count. Duration 30 microseconds. Forward packets 1. Backward packets 1. Flow bytes per second 266666.67. TCP flags: none.
Network flow to unknown service on port 3844. very short connection. low packet count. Duration 29 microseconds. Forward packets 1. Backward packets 1. Flow bytes per second 413793.1. TCP flags: ACK.
Network flow to unknown service on port 84. very short connection. low packet count. Duration 70 microseconds. Forward packets 1

In [121]:
# Community summary
summary_rows = []

for community_id, group in community_df.groupby("community_id"):
    dominant_tactic = group["attck_tactic"].mode().iloc[0]
    dominant_label = group["Label"].mode().iloc[0]
    homogeneity = group["attck_tactic"].value_counts(normalize=True).iloc[0]

    summary_rows.append({
        "community_id": community_id,
        "size": len(group),
        "dominant_label": dominant_label,
        "dominant_tactic": dominant_tactic,
        "homogeneity": round(homogeneity, 4)
    })

summary_df = pd.DataFrame(summary_rows).sort_values("size", ascending=False)
summary_df.to_csv(SUMMARY_FILE, index=False)

print(f"Saved to {SUMMARY_FILE}")
summary_df.head(10)

Saved to ../data/results/community_summary.csv


,community_id,size,dominant_label,dominant_tactic,homogeneity
0,0,3188,PortScan,Discovery,0.6066
5,5,3057,DDoS,Impact,0.6542
7,7,1268,Bot,Command And Control,0.9937
1,1,1095,FTP Patator,Credential Access,0.9909
2,2,822,BENIGN,Benign,0.9976
8,8,735,SSH Patator,Credential Access,0.9850
6,6,431,BENIGN,Benign,0.9977
11,15,17,PortScan,Discovery,0.8235
15,21,12,Bot,Command And Control,0.7500
10,11,10,PortScan,Discovery,0.7000


In [122]:
# Intra-cluster similarity
cluster_sim_rows = []

for community_id, group in community_df.groupby("community_id"):
    ids = group["sample_id"].tolist()
    if len(ids) < 2:
        mean_sim = 1.0
    else:
        sims = similarity_matrix[np.ix_(ids, ids)]
        upper = sims[np.triu_indices_from(sims, k=1)]
        mean_sim = float(upper.mean()) if len(upper) > 0 else 1.0

    cluster_sim_rows.append({
        "community_id": community_id,
        "mean_intra_similarity": round(mean_sim, 4)
    })

cluster_sim_df = pd.DataFrame(cluster_sim_rows)
summary_df = summary_df.merge(cluster_sim_df, on="community_id", how="left")
summary_df.to_csv(SUMMARY_FILE, index=False)
summary_df.head(10)

,community_id,size,dominant_label,dominant_tactic,homogeneity,mean_intra_similarity
0,0,3188,PortScan,Discovery,0.6066,0.8623
1,5,3057,DDoS,Impact,0.6542,0.9486
2,7,1268,Bot,Command And Control,0.9937,0.9751
3,1,1095,FTP Patator,Credential Access,0.9909,0.9463
4,2,822,BENIGN,Benign,0.9976,0.9651
5,8,735,SSH Patator,Credential Access,0.9850,0.9366
6,6,431,BENIGN,Benign,0.9977,0.9494
7,15,17,PortScan,Discovery,0.8235,0.9682
8,21,12,Bot,Command And Control,0.7500,0.9723
9,11,10,PortScan,Discovery,0.7000,0.9760


In [123]:
# Save clustering metrics
clustering_metrics = {
    "threshold": SIMILARITY_THRESHOLD,
    "min_community_size": MIN_COMMUNITY_SIZE,
    "n_communities": int(summary_df["community_id"].nunique()),
    "mean_homogeneity": float(summary_df["homogeneity"].mean()),
    "median_homogeneity": float(summary_df["homogeneity"].median()),
    "mean_intra_similarity": float(summary_df["mean_intra_similarity"].mean()),
    "median_intra_similarity": float(summary_df["mean_intra_similarity"].median()),
    "mean_community_size": float(summary_df["size"].mean()),
    "median_community_size": float(summary_df["size"].median())
}

with open(CLUSTER_METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(clustering_metrics, f, indent=2)

print(f"Saved to {CLUSTER_METRICS_FILE}")
clustering_metrics

Saved to ../data/results/clustering_metrics.json


{'threshold': 0.9,
 'min_community_size': 3,
 'n_communities': 16,
 'mean_homogeneity': 0.847175,
 'median_homogeneity': 0.90425,
 'mean_intra_similarity': 0.96081875,
 'median_intra_similarity': 0.96665,
 'mean_community_size': 666.375,
 'median_community_size': 14.5}

In [124]:
# Build text blocks for each community to feed into LLM
community_texts = {}

for community_id, group in community_df.groupby("community_id"):
    texts = group["alert_text"].dropna().tolist()[:MAX_ALERTS_PER_COMMUNITY]
    text_block = "\n".join([f"- {t}" for t in texts])
    community_texts[str(community_id)] = text_block

print("Prepared text blocks for", len(community_texts), "communities")

Prepared text blocks for 16 communities


In [125]:
def build_triple_prompt(text_block):
    return f"""
You are extracting structured knowledge from grouped SIEM alert descriptions.

Task:
Extract 4 to 6 useful triples that summarize the overall behavior of the alert community.

Return only a JSON array.
Do not explain anything.
Do not use markdown.
Do not return any text before or after the JSON.

Use this exact format:
[
  {{"subject": "...", "relation": "...", "object": "..."}}
]

Rules:
- Focus on the dominant behavior of the whole community, not each individual alert.
- Prefer meaningful semantic summaries over repeating raw numbers.
- Avoid duplicate triples.
- Use lowercase text.
- Use short, consistent relation names.
- Use "network flow" as the default subject only when no clearer subject is available.
- If a clearer subject exists, you may use terms like "ftp service", "ssh service", "http service", or "traffic pattern".
- If nothing useful is found, return [].
- Do not exaggerate.
- Only output information supported by the alert text.

Allowed relations:
- "targets_service"
- "targets_port"
- "shows_flag"
- "shows_behavior"
- "has_duration"
- "has_packet_count"
- "indicates_activity"

Allowed activity objects for "indicates_activity":
- "port scanning activity"
- "brute force login activity"
- "denial of service behavior"
- "web protocol communication"
- "tool transfer behavior"

How to summarize:
- Use "short duration" or "long duration" instead of exact microseconds.
- Use "low packet count" or "high packet count" instead of exact counts.
- Use behavior labels such as:
  "observed traffic pattern"
  "very short connection"
  "high volume traffic"
  "incomplete handshake pattern"

Examples:
[
  {{"subject": "network flow", "relation": "targets_service", "object": "http service"}},
  {{"subject": "network flow", "relation": "shows_behavior", "object": "very short connection"}},
  {{"subject": "network flow", "relation": "has_duration", "object": "short duration"}},
  {{"subject": "network flow", "relation": "indicates_activity", "object": "port scanning activity"}}
]

[
  {{"subject": "network flow", "relation": "targets_service", "object": "ftp service"}},
  {{"subject": "network flow", "relation": "has_packet_count", "object": "low packet count"}},
  {{"subject": "network flow", "relation": "shows_behavior", "object": "observed traffic pattern"}},
  {{"subject": "network flow", "relation": "indicates_activity", "object": "brute force login activity"}}
]

Text:
{text_block}
"""

In [126]:
# Call Ollama 
def call_ollama(prompt, model=MODEL_NAME):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
    response.raise_for_status()

    data = response.json()
    return data.get("response", "").strip()

In [127]:
# Parse triples from LLM response
def parse_triples(raw_text):
    try:
        data = json.loads(raw_text)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

In [128]:
# Clean and filter triples
def clean_triples(output):
    if not isinstance(output, list):
        return []

    clean = []
    seen = set()

    for item in output:
        if not isinstance(item, dict):
            continue

        subject = str(item.get("subject", "")).strip().lower()
        relation = str(item.get("relation", "")).strip().lower()
        obj = str(item.get("object", "")).strip().lower()

        if subject == "network_flow":
            subject = "network flow"

        if not subject or not relation or not obj:
            continue

        # Split combined flags like "syn, ack"
        if relation == "shows_flag" and "," in obj:
            parts = [p.strip() for p in obj.split(",") if p.strip()]
            for part in parts:
                key = (subject, relation, part)
                if key not in seen:
                    seen.add(key)
                    clean.append({
                        "subject": subject,
                        "relation": relation,
                        "object": part
                    })
            continue

        key = (subject, relation, obj)
        if key in seen:
            continue

        seen.add(key)

        clean.append({
            "subject": subject,
            "relation": relation,
            "object": obj
        })

    return clean

In [129]:
# Test one community
test_id = list(community_texts.keys())[0]
raw_output = call_ollama(build_triple_prompt(community_texts[test_id]))

print("RAW OUTPUT:")
print(raw_output)

parsed_output = parse_triples(raw_output)
print("\nPARSED OUTPUT:")
print(parsed_output)

clean_output = clean_triples(parsed_output)
print("\nCLEAN OUTPUT:")
print(clean_output)

RAW OUTPUT:
[
  {"subject": "network flow", "relation": "targets_service", "object": "unknown service"},
  {"subject": "network flow", "relation": "shows_behavior", "object": "very short connection"},
  {"subject": "network flow", "relation": "has_duration", "object": "short duration"},
  {"subject": "network flow", "relation": "indicates_activity", "object": "port scanning activity"}
]

PARSED OUTPUT:
[{'subject': 'network flow', 'relation': 'targets_service', 'object': 'unknown service'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'very short connection'}, {'subject': 'network flow', 'relation': 'has_duration', 'object': 'short duration'}, {'subject': 'network flow', 'relation': 'indicates_activity', 'object': 'port scanning activity'}]

CLEAN OUTPUT:
[{'subject': 'network flow', 'relation': 'targets_service', 'object': 'unknown service'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'very short connection'}, {'subject': 'network flow',

In [130]:
# Run for all communities
community_triples = {}
failed_communities = []

for community_id, text_block in community_texts.items():
    prompt = build_triple_prompt(text_block)

    try:
        raw_output = call_ollama(prompt)
        parsed_output = parse_triples(raw_output)
        triples = clean_triples(parsed_output)
    except Exception as e:
        print(f"Error in community {community_id}: {e}")
        triples = []
        failed_communities.append(community_id)

    community_triples[community_id] = triples

In [131]:
# Inspect results,
for cid in list(community_triples.keys())[:5]:
    print(f"\nCommunity {cid}")
    print(community_triples[cid])


Community 0
[{'subject': 'network flow', 'relation': 'targets_service', 'object': 'unknown service'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'very short connection'}, {'subject': 'network flow', 'relation': 'has_duration', 'object': 'short duration'}, {'subject': 'network flow', 'relation': 'indicates_activity', 'object': 'port scanning activity'}]

Community 1
[{'subject': 'network flow', 'relation': 'targets_service', 'object': 'ftp service'}, {'subject': 'network flow', 'relation': 'has_duration', 'object': 'long duration'}, {'subject': 'network flow', 'relation': 'indicates_activity', 'object': 'port scanning activity'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'observed traffic pattern'}]

Community 2
[{'subject': 'network flow', 'relation': 'targets_service', 'object': 'dns service'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'observed traffic pattern'}, {'subject': 'network flow', 'relation': 'has

In [132]:
# manually review a sample of communities and their triples
review_rows = []

top_communities = (
    community_df.groupby("community_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

for cid in top_communities:
    group = community_df[community_df["community_id"] == cid]
    review_rows.append({
        "community_id": cid,
        "size": len(group),
        "dominant_label": group["Label"].mode().iloc[0],
        "dominant_tactic": group["attck_tactic"].mode().iloc[0],
        "sample_alert_text": group["alert_text"].iloc[0],
        "triples": json.dumps(community_triples.get(str(cid), []), ensure_ascii=False),
        "semantic_alignment": ""   # fill manually: fully_aligned / partially_aligned / misaligned
    })

semantic_review_df = pd.DataFrame(review_rows)
semantic_review_df.to_csv(OUTPUT_DIR / "semantic_review_sample.csv", index=False)

semantic_review_df.head(10)


,community_id,size,dominant_label,dominant_tactic,sample_alert_text,triples,semantic_alignment
0,0,3188,PortScan,Discovery,Network flow to unknown service on port 1123. ...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
1,5,3057,DDoS,Impact,Network flow to HTTP service. long duration. D...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
2,7,1268,Bot,Command And Control,Network flow to HTTP-alt service. observed tra...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
3,1,1095,FTP Patator,Credential Access,Network flow to FTP service. long duration. Du...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
4,2,822,BENIGN,Benign,Network flow to DNS service. high volume traff...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
5,8,735,SSH Patator,Credential Access,Network flow to SSH service. long duration. Du...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
6,6,431,BENIGN,Benign,Network flow to HTTPS service. observed traffi...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
7,15,17,PortScan,Discovery,Network flow to unknown service on port 5906. ...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
8,21,12,Bot,Command And Control,Network flow to unknown service on port 2968. ...,"[{""subject"": ""network flow"", ""relation"": ""targ...",
9,11,10,PortScan,Discovery,Network flow to unknown service on port 3382. ...,"[{""subject"": ""network flow"", ""relation"": ""targ...",


In [133]:
# keyword-based support check for each triple against the community text
def keyword_supported(triple, text_block):
    subject = triple.get("subject", "").lower()
    relation = triple.get("relation", "").lower()
    obj = triple.get("object", "").lower()
    text = text_block.lower()

    checks = []

    if obj:
        checks.append(obj in text)

    # optional relation-specific hints
    if relation == "targets_port" and obj.isdigit():
        checks.append(f"port {obj}" in text)

    if relation == "shows_flag":
        checks.append(obj.upper() in text.upper())

    return any(checks) if checks else False


support_rows = []

for cid, triples in community_triples.items():
    text_block = community_texts.get(cid, "")
    n_triples = len(triples)
    supported = sum(keyword_supported(t, text_block) for t in triples)

    support_rows.append({
        "community_id": cid,
        "n_triples": n_triples,
        "n_supported": supported,
        "support_ratio": supported / n_triples if n_triples > 0 else 0.0
    })

support_df = pd.DataFrame(support_rows).sort_values("support_ratio", ascending=False)
support_df.head(10)

,community_id,n_triples,n_supported,support_ratio
1,1,4,3,0.75
5,5,4,3,0.75
2,2,4,2,0.50
0,0,4,2,0.50
3,3,4,2,0.50
4,4,4,2,0.50
6,6,4,2,0.50
7,7,4,2,0.50
8,8,4,2,0.50
9,10,4,2,0.50


In [134]:
semantic_support_metrics = {
    "mean_support_ratio": float(support_df["support_ratio"].mean()),
    "median_support_ratio": float(support_df["support_ratio"].median()),
    "n_communities": int(len(support_df))
}

print(semantic_support_metrics)

{'mean_support_ratio': 0.53125, 'median_support_ratio': 0.5, 'n_communities': 16}


In [135]:
# Triple extraction metrics
triple_rows = []

for cid, triples in community_triples.items():
    n_triples = len(triples)
    valid = sum(
        1 for t in triples
        if isinstance(t, dict)
        and all(k in t for k in ["subject", "relation", "object"])
        and t["subject"] and t["relation"] and t["object"]
    )

    triple_rows.append({
        "community_id": cid,
        "n_triples": n_triples,
        "valid_triples": valid
    })

triple_eval_df = pd.DataFrame(triple_rows)
triple_eval_df["valid_ratio"] = (
    triple_eval_df["valid_triples"] / triple_eval_df["n_triples"].replace(0, np.nan)
).fillna(0)

triple_eval_df.head()

,community_id,n_triples,valid_triples,valid_ratio
0,0,4,4,1.0
1,1,4,4,1.0
2,2,4,4,1.0
3,3,4,4,1.0
4,4,4,4,1.0


In [136]:
# Coverage = communities with at least one usable triple
coverage = float((triple_eval_df["n_triples"] > 0).mean())

triple_metrics = {
    "n_communities": int(len(triple_eval_df)),
    "coverage": coverage,
    "mean_triples_per_community": float(triple_eval_df["n_triples"].mean()),
    "median_triples_per_community": float(triple_eval_df["n_triples"].median()),
    "mean_valid_ratio": float(triple_eval_df["valid_ratio"].mean()),
    "failed_communities": len(failed_communities)
}

with open(TRIPLE_METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(triple_metrics, f, indent=2)

print(f"Saved to {TRIPLE_METRICS_FILE}")
triple_metrics

Saved to ../data/results/triple_metrics.json


{'n_communities': 16,
 'coverage': 1.0,
 'mean_triples_per_community': 4.0,
 'median_triples_per_community': 4.0,
 'mean_valid_ratio': 1.0,
 'failed_communities': 0}

In [137]:
for cid in list(community_triples.keys())[:3]:
    print(f"\nCommunity {cid}")
    print("Triples:", community_triples[cid])
    print("\nSample alerts:")
    print(community_texts[cid][:300])


Community 0
Triples: [{'subject': 'network flow', 'relation': 'targets_service', 'object': 'unknown service'}, {'subject': 'network flow', 'relation': 'shows_behavior', 'object': 'very short connection'}, {'subject': 'network flow', 'relation': 'has_duration', 'object': 'short duration'}, {'subject': 'network flow', 'relation': 'indicates_activity', 'object': 'port scanning activity'}]

Sample alerts:
- Network flow to unknown service on port 1123. very short connection. low packet count. Duration 38 microseconds. Forward packets 1. Backward packets 1. Flow bytes per second 210526.32. TCP flags: none.
- Network flow to unknown service on port 1580. very short connection. Duration 50 microseconds.

Community 1
Triples: [{'subject': 'network flow', 'relation': 'targets_service', 'object': 'ftp service'}, {'subject': 'network flow', 'relation': 'has_duration', 'object': 'long duration'}, {'subject': 'network flow', 'relation': 'indicates_activity', 'object': 'port scanning activity'}, {'

In [138]:
triple_eval_df["n_triples"].describe()

count    16.0
mean      4.0
std       0.0
min       4.0
25%       4.0
50%       4.0
75%       4.0
max       4.0
Name: n_triples, dtype: float64

In [139]:
# Save triples
with open(TRIPLES_FILE, "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)

print(f"Saved to {TRIPLES_FILE}")

Saved to ../data/results/community_triples.json
